# Dhruv Patel Academy blank-page practice: Uncertainty: Bayes, Calibration & Conformal Prediction

**Blank-page version:** every code cell is empty - write the whole answer yourself. (The fill-in-the-blank version is `uncertainty.ipynb`.)

Practical questions from the **Uncertainty: Bayes, Calibration & Conformal Prediction** module. Write your answer in the code cell under each question, run it, then run the **check** cell right below it. The check runs *in this notebook*, on your own variables, and tells you exactly what is right or wrong.

1. Run the first code cell once (it loads the checker and the 12 questions).
2. Work through the questions in any order.
3. `dsq.hint("question-id")` gives a hint; `dsq.solution("question-id")` shows the reference answer (only after you tried 3 times).
4. `dsq.score()` shows where you stand.

Optional: `dsq.connect("http://127.0.0.1:5177", "YOUR-KEY")` (key from the module page in Dhruv Patel Academy) sends solved questions to the app so your profile counts them.

You need Python with `numpy`, `pandas`, `matplotlib` and `seaborn`, e.g. `pip install numpy pandas matplotlib seaborn notebook`.

In [ ]:
# --- Dhruv Patel Academy practice checker (runs in YOUR notebook, on YOUR variables) ---
import json as _json, sys as _sys, urllib.request as _ur

class _DSQ:
    def __init__(self):
        self.qs = {}; self.order = []; self.tries = {}; self.ok = {}; self.url = None; self.key = None; self.module = ''

    def load(self, pack):
        self.module = pack['title']
        for q in pack['questions']:
            self.qs[q['id']] = q; self.order.append(q['id'])
        print(f"Dhruv Patel Academy: {len(self.order)} questions loaded for '{self.module}'. Write your answer, then run the check cell under it.")
        if self._auto(): print('Linked to your Dhruv Patel Academy account: solved questions count in the app.')

    def connect(self, url, key, quiet=False):
        self.url = url.rstrip('/'); self.key = key
        if not quiet: print('Connected: solved questions will be reported to', self.url)

    def _auto(self):
        # Dhruv Patel Academy writes dsq_config.json next to the notebooks when you open them from the app
        import os
        d = os.getcwd()
        for _ in range(4):
            p = os.path.join(d, 'dsq_config.json')
            if os.path.exists(p):
                try:
                    c = _json.load(open(p)); self.connect(c['url'], c['key'], quiet=True); return True
                except Exception:
                    pass
            d = os.path.dirname(d)
        return False

    def _report(self, qid, ok):
        if not self.url: return
        try:
            req = _ur.Request(self.url + '/api/nb/report', data=_json.dumps({'key': self.key, 'qid': qid, 'ok': bool(ok)}).encode(), headers={'Content-Type': 'application/json'})
            _ur.urlopen(req, timeout=4).read()
        except Exception as e:
            print('(could not reach the app:', str(e)[:80] + ')')

    def check(self, qid):
        q = self.qs[qid]
        try:
            ip = get_ipython(); ns = dict(ip.user_ns)
        except NameError:
            ns = dict(_sys.modules['__main__'].__dict__)
        results = []
        def check(cond, msg, hint=''):
            ok = bool(cond); results.append((ok, msg, hint)); return ok
        ns['check'] = check; ns['OUT'] = ''
        for mod, alias in (('numpy', 'np'), ('pandas', 'pd')):
            if alias not in ns:
                try: ns[alias] = __import__(mod)
                except Exception: pass
        if 'plt' not in ns:
            try:
                import matplotlib.pyplot as plt; ns['plt'] = plt
            except Exception: pass
        try:
            exec(compile(q['tests'], '<tests>', 'exec'), ns)
        except BaseException as e:
            results.append((False, 'A check could not finish: ' + type(e).__name__ + ': ' + str(e)[:200], 'Did you create every variable name the task asks for? Did you run your answer cell first?'))
        self.tries[qid] = self.tries.get(qid, 0) + 1
        good = bool(results) and all(r[0] for r in results)
        for ok, msg, hint in results:
            print(('  [PASS] ' if ok else '  [FAIL] ') + msg + ('' if ok or not hint else '\n         hint: ' + hint))
        if good:
            self.ok[qid] = True; print(f"\n  Question solved: {q['title']}  (attempt {self.tries[qid]})")
        else:
            print(f"\n  Not yet - fix your answer cell, run it again, then re-run this check. (attempt {self.tries[qid]})")
        self._report(qid, good)
        return good

    def hint(self, qid):
        hs = self.qs[qid].get('hints') or []
        n = min(self.tries.get(qid, 0), len(hs)) if hs else 0
        print(('hint: ' + hs[max(0, n - 1)]) if hs else 'No hint for this one - re-read the question.')

    def solution(self, qid):
        if self.tries.get(qid, 0) < 3 and not self.ok.get(qid):
            print('Try the check at least 3 times first - the struggle is where the learning happens.'); return
        print(self.qs[qid]['solution'])

    def score(self):
        n = len(self.order); s = sum(1 for q in self.order if self.ok.get(q))
        print(f"Solved {s} of {n}.")
        for q in self.order:
            print(('  [x] ' if self.ok.get(q) else '  [ ] ') + q + ' - ' + self.qs[q]['title'] + ('' if self.ok.get(q) else f"  (tries: {self.tries.get(q, 0)})"))

dsq = _DSQ()


dsq.load({
 "module": "uncertainty",
 "title": "Uncertainty: Bayes, Calibration & Conformal Prediction",
 "questions": [
  {
   "id": "un-1-1",
   "title": "Update a Beta prior",
   "tests": "check(beta_update(1, 1, 7, 10) == (8, 4), '7 clicks, 3 non-clicks added to a flat prior', 'a + k, b + n - k')\ncheck(abs(beta_mean(8, 4) - 2/3) < 1e-12, 'the posterior mean is 0.667, not the raw 0.70: the flat prior pulls slightly', 'a / (a + b)')\ncheck(abs(shrunk_rate(1, 1, 0.2, 20) - 5/21) < 1e-12 and shrunk_rate(1, 1, 0.2, 20) < 0.25, 'one success in one try does not mean 100%: with a strong prior it stays near 24%', 'prior pseudo-counts')",
   "hints": [
    "Successes add to a, failures add to b."
   ],
   "solution": "def beta_update(a, b, k, n):\n    return a + k, b + n - k\n\ndef beta_mean(a, b):\n    return a / (a + b)\n\ndef shrunk_rate(k, n, prior_mean, prior_n):\n    return (k + prior_mean * prior_n) / (n + prior_n)\n\nprint(beta_update(1, 1, 7, 10), beta_mean(8, 4), shrunk_rate(1, 1, 0.2, 20))"
  },
  {
   "id": "un-1-2",
   "title": "A credible interval from a grid",
   "tests": "lo, hi = credible_interval(1, 1)\ncheck(abs(lo - 0.025) < 0.002 and abs(hi - 0.975) < 0.002, 'a flat prior: the middle 95% runs from 0.025 to 0.975', 'cumulative probability 2.5% and 97.5%')\nlo, hi = credible_interval(8, 4)\ncheck(lo < 8 / 12 < hi and 0.38 < lo < 0.42 and 0.88 < hi < 0.94, 'after 7 of 10 clicks the rate is probably between about 0.4 and 0.9', 'the mean lies inside')\nw_small = credible_interval(3, 3)\nw_big = credible_interval(30, 30)\ncheck((w_big[1] - w_big[0]) < (w_small[1] - w_small[0]) / 2, 'ten times more data gives a much narrower interval', 'uncertainty shrinks with data')",
   "hints": [
    "Normalise first, then walk along the grid with a running total."
   ],
   "solution": "import math\n\ndef credible_interval(a, b, level=0.95, grid=2001):\n    xs = [(i + 0.5) / grid for i in range(grid)]\n    w = [math.exp((a - 1) * math.log(x) + (b - 1) * math.log(1 - x)) for x in xs]\n    s = sum(w)\n    lo_t, hi_t = (1 - level) / 2, 1 - (1 - level) / 2\n    acc, lo, hi = 0.0, None, None\n    for x, wi in zip(xs, w):\n        acc += wi / s\n        if lo is None and acc >= lo_t:\n            lo = x\n        if hi is None and acc >= hi_t:\n            hi = x\n            break\n    return lo, hi\n\nprint(credible_interval(1, 1), credible_interval(8, 4))"
  },
  {
   "id": "un-1-3",
   "title": "Thompson sampling",
   "tests": "check(all(thompson_pick([(100, 1), (1, 100)], s) == 0 for s in range(50)), 'a clearly better arm is always chosen', 'draws from Beta(100, 1) are near 1')\npicks = [thompson_pick([(1, 1), (1, 1)], s) for s in range(400)]\ncheck(150 < picks.count(0) < 250, 'two equally unknown arms are picked about equally often', 'exploration when uncertain')\npicks = [thompson_pick([(30, 70), (2, 1)], s) for s in range(400)]\ncheck(picks.count(1) > 100, 'an arm with a good but very uncertain record still gets tried a lot', 'wide posterior = frequent exploration')",
   "hints": [
    "One draw per arm, then argmax."
   ],
   "solution": "import random\n\ndef thompson_pick(arms, seed):\n    rng = random.Random(seed)\n    draws = [rng.betavariate(a, b) for a, b in arms]\n    return draws.index(max(draws))\n\nprint(thompson_pick([(100, 1), (1, 100)], 0))"
  },
  {
   "id": "un-2-1",
   "title": "Bootstrap any statistic",
   "tests": "import statistics\nx = [3, 5, 4, 10, 6, 5, 4, 7, 5, 30]\nlo, hi = bootstrap_ci(x, statistics.median)\ncheck(lo <= statistics.median(x) <= hi and lo >= 3 and hi <= 10, 'a median interval stays among the typical values, unaffected by the outlier 30', 'percentiles of resampled medians')\nlm, hm = bootstrap_ci(x, statistics.mean)\ncheck((hm - lm) > (hi - lo), 'the mean is far more affected by the outlier, so its interval is wider', 'same method, different statistic')\nc = bootstrap_ci([2.0] * 20, max)\ncheck(c == (2.0, 2.0), 'no variation, no uncertainty, for any statistic', 'identical resamples')",
   "hints": [
    "stat is any function that takes a list and returns a number."
   ],
   "solution": "import random, statistics\n\ndef bootstrap_ci(x, stat, B=2000, alpha=0.05, seed=0):\n    rng = random.Random(seed)\n    vals = sorted(stat(rng.choices(x, k=len(x))) for _ in range(B))\n    return vals[int(B * alpha / 2)], vals[int(B * (1 - alpha / 2)) - 1]\n\nprint(bootstrap_ci([3, 5, 4, 10, 6, 5, 4, 7, 5, 30], statistics.median))"
  },
  {
   "id": "un-2-2",
   "title": "Does a 95% interval cover 95%?",
   "tests": "c = coverage(10, 2, 30, 2000, 0)\ncheck(0.92 < c < 0.97, 'with n = 30 about 94-95% of the intervals contain the truth', 'a well-behaved 95% interval')\nc2 = coverage(10, 2, 5, 2000, 0)\ncheck(c2 < c, 'with only 5 observations the 1.96 rule is too optimistic: coverage drops (about 88%)', 'small n needs a t multiplier')\ncheck(coverage(0, 1, 30, 300, 4) == coverage(0, 1, 30, 300, 4), 'the same seed repeats the same experiment', 'the generator is built inside')",
   "hints": [
    "Count the trials where the interval catches the true mean."
   ],
   "solution": "import random, statistics, math\n\ndef coverage(true_mean, sd, n, trials, seed):\n    rng = random.Random(seed)\n    hits = 0\n    for _ in range(trials):\n        xs = [rng.gauss(true_mean, sd) for _ in range(n)]\n        m, s = statistics.mean(xs), statistics.stdev(xs)\n        half = 1.96 * s / math.sqrt(n)\n        if m - half <= true_mean <= m + half:\n            hits += 1\n    return hits / trials\n\nprint(coverage(10, 2, 30, 2000, 0))"
  },
  {
   "id": "un-2-3",
   "title": "How many samples do I need?",
   "tests": "check(abs(margin(10, 100) - 1.96) < 1e-12, 'sd 10 with 100 observations: a margin of 1.96', 'z sd / sqrt(n)')\ncheck(n_for_margin(10, 1) == 385 and n_for_margin(10, 0.5) == 1537, 'to halve the margin you need about four times the data: 385 becomes 1,537', 'ceil((z sd / target)^2)')\ncheck(margin(10, n_for_margin(10, 1)) <= 1.0 < margin(10, n_for_margin(10, 1) - 1), 'n is the smallest value that works', 'one fewer would miss the target')",
   "hints": [
    "Solve margin = z sd / sqrt(n) for n."
   ],
   "solution": "import math\n\ndef margin(sd, n, z=1.96):\n    return z * sd / math.sqrt(n)\n\ndef n_for_margin(sd, target, z=1.96):\n    return math.ceil((z * sd / target) ** 2)\n\nprint(margin(10, 100), n_for_margin(10, 1))"
  },
  {
   "id": "un-3-1",
   "title": "Expected calibration error",
   "tests": "check(abs(ece([0.9] * 10, [1] * 6 + [0] * 4) - 0.3) < 1e-12, 'says 90%, right 60% of the time: ECE 0.3', '|acc - conf| in the single bin')\ncheck(abs(ece([0.5] * 10, [1] * 5 + [0] * 5)) < 1e-12 and ece([1.0, 1.0], [1, 1]) == 0.0, 'perfectly calibrated bins (and a probability of exactly 1.0) give 0', 'last bin includes 1.0')\ncheck(abs(ece([0.1] * 5 + [0.9] * 5, [0] * 4 + [1] + [1] * 4 + [0]) - 0.1) < 1e-12, 'two bins, both off by 0.1', 'weighted average of the gaps')",
   "hints": [
    "int(p * bins) gives the bin; clip it with min(..., bins - 1)."
   ],
   "solution": "def ece(probs, labels, bins=10):\n    n = len(probs)\n    groups = [[] for _ in range(bins)]\n    for p, y in zip(probs, labels):\n        i = min(int(p * bins), bins - 1)\n        groups[i].append((p, y))\n    total = 0.0\n    for g in groups:\n        if g:\n            conf = sum(p for p, _ in g) / len(g)\n            acc = sum(y for _, y in g) / len(g)\n            total += len(g) / n * abs(acc - conf)\n    return total\n\nprint(ece([0.9] * 10, [1] * 6 + [0] * 4))"
  },
  {
   "id": "un-3-2",
   "title": "A reliability table",
   "tests": "t = reliability([0.1, 0.15, 0.9, 0.95, 0.85], [0, 1, 1, 1, 0])\ncheck(len(t) == 2 and t[0][:3] == (0.0, 0.2, 2) and t[1][2] == 3, 'two non-empty bins holding 2 and 3 predictions', 'only bins with data appear')\ncheck(abs(t[0][3] - 0.125) < 1e-12 and abs(t[0][4] - 0.5) < 1e-12, 'the low bin predicts 12.5% but 50% are positive: under-confident', 'mean prob versus fraction positive')\ncheck(abs(t[1][3] - 0.9) < 1e-12 and abs(t[1][4] - 2/3) < 1e-12, 'the high bin predicts 90% but only 67% are positive: over-confident', 'compare the two numbers')",
   "hints": [
    "Skip empty bins."
   ],
   "solution": "def reliability(probs, labels, bins=5):\n    groups = [[] for _ in range(bins)]\n    for p, y in zip(probs, labels):\n        groups[min(int(p * bins), bins - 1)].append((p, y))\n    out = []\n    for i, g in enumerate(groups):\n        if g:\n            out.append((i / bins, (i + 1) / bins, len(g), sum(p for p, _ in g) / len(g), sum(y for _, y in g) / len(g)))\n    return out\n\nprint(reliability([0.1, 0.15, 0.9, 0.95, 0.85], [0, 1, 1, 1, 0]))"
  },
  {
   "id": "un-3-3",
   "title": "Find the temperature",
   "tests": "import math, random\ncheck(abs(nll([2.0, -2.0], [1, 0], 1.0) - math.log(1 + math.exp(-2))) < 1e-12, 'the NLL of two confident correct predictions', '-log sigmoid(2)')\nrng = random.Random(0)\nz = [rng.gauss(0, 3) for _ in range(6000)]\ny = [1 if rng.random() < 1 / (1 + math.exp(-v / 2)) else 0 for v in z]\ngrid = [0.5 + 0.1 * i for i in range(36)]\nT = best_temperature(z, y, grid)\ncheck(1.7 <= T <= 2.3, 'the logits are twice too confident: the best temperature is about 2', 'minimise the held-out NLL')\ncheck(nll(z, y, T) < nll(z, y, 1.0), 'scaling by the best T lowers the loss', 'a better-calibrated model')",
   "hints": [
    "Loop over the grid and keep the smallest NLL."
   ],
   "solution": "import math\n\ndef nll(logits, labels, T):\n    total = 0.0\n    for z, y in zip(logits, labels):\n        p = 1 / (1 + math.exp(-z / T))\n        p = min(max(p, 1e-12), 1 - 1e-12)\n        total += -(y * math.log(p) + (1 - y) * math.log(1 - p))\n    return total / len(logits)\n\ndef best_temperature(logits, labels, grid):\n    best, arg = float(\"inf\"), grid[0]\n    for T in grid:\n        v = nll(logits, labels, T)\n        if v < best - 1e-15:\n            best, arg = v, T\n    return arg\n\nprint(nll([2.0, -2.0], [1, 0], 1.0))"
  },
  {
   "id": "un-4-1",
   "title": "The conformal quantile",
   "tests": "check(conformal_q(list(range(1, 20)), 0.1) == 18, '19 scores at alpha = 0.1: the 18th smallest', 'k = ceil(20 x 0.9) = 18')\ncheck(conformal_q([1, 2, 3, 4, 5], 0.1) == float('inf'), 'with 5 calibration points you cannot promise 90% (k = 6 > 5): the interval is infinite', 'k > n')\ncheck(conformal_q([5, 1, 4, 2, 3, 9, 7, 8, 6, 10], 0.2) == 9, 'order does not matter; 10 scores at 0.2 give the 9th smallest', 'sort first')",
   "hints": [
    "k counts from 1, so use index k - 1."
   ],
   "solution": "import math\n\ndef conformal_q(scores, alpha):\n    n = len(scores)\n    k = math.ceil((n + 1) * (1 - alpha))\n    if k > n:\n        return float(\"inf\")\n    return sorted(scores)[k - 1]\n\nprint(conformal_q(list(range(1, 20)), 0.1), conformal_q([1, 2, 3, 4, 5], 0.1))"
  },
  {
   "id": "un-4-2",
   "title": "Intervals and their coverage",
   "tests": "check(interval(10.0, 2.5) == (7.5, 12.5), 'prediction plus or minus q', 'pred - q, pred + q')\ncheck(abs(empirical_coverage([1, 5, 9], [2, 5, 4], 1.5) - 2/3) < 1e-12, 'two of three true values fall inside', '|y - pred| <= q')\ncheck(empirical_coverage([1, 2], [1, 2], 0) == 1.0 and empirical_coverage([0], [5], 4.9) == 0.0, 'a zero-width interval still covers exact predictions; a miss is a miss', 'inclusive comparison')",
   "hints": [
    "A case is covered when its error is at most q."
   ],
   "solution": "def interval(pred, q):\n    return pred - q, pred + q\n\ndef empirical_coverage(ys, preds, q):\n    return sum(1 for y, p in zip(ys, preds) if abs(y - p) <= q) / len(ys)\n\nprint(interval(10.0, 2.5), empirical_coverage([1, 5, 9], [2, 5, 4], 1.5))"
  },
  {
   "id": "un-4-3",
   "title": "Split conformal end to end",
   "tests": "import random\nrng = random.Random(0)\nf = lambda x: 2 * x\nxc = [rng.uniform(0, 10) for _ in range(500)]; yc = [2 * x + rng.gauss(0, 1) for x in xc]\nxt = [rng.uniform(0, 10) for _ in range(5000)]; yt = [2 * x + rng.gauss(0, 1) for x in xt]\nq, cov = split_conformal(xc, yc, xt, yt, 0.1, f)\ncheck(1.5 < q < 1.8, 'for noise with sd 1 the 90% half-width is about 1.645', 'the (n+1)(1-alpha)-th smallest error')\ncheck(0.87 < cov < 0.93, 'about 90% of fresh test cases fall inside, as promised', 'the coverage guarantee')\nq2, cov2 = split_conformal(xc, yc, xt, yt, 0.2, f)\ncheck(q2 < q and 0.74 < cov2 < 0.84, 'asking for only 80% gives a narrower interval and about 80% coverage', 'larger alpha, smaller q')",
   "hints": [
    "Calibration scores come from the calibration set only; coverage is measured on fresh test data."
   ],
   "solution": "import math, random\n\ndef conformal_q(scores, alpha):\n    n = len(scores); k = math.ceil((n + 1) * (1 - alpha))\n    return float(\"inf\") if k > n else sorted(scores)[k - 1]\n\ndef split_conformal(xc, yc, xt, yt, alpha, f):\n    scores = [abs(y - f(x)) for x, y in zip(xc, yc)]\n    q = conformal_q(scores, alpha)\n    cov = sum(1 for x, y in zip(xt, yt) if abs(y - f(x)) <= q) / len(xt)\n    return q, cov\n\nrng = random.Random(0)\nf = lambda x: 2 * x\nxc = [rng.uniform(0, 10) for _ in range(500)]; yc = [2 * x + rng.gauss(0, 1) for x in xc]\nxt = [rng.uniform(0, 10) for _ in range(5000)]; yt = [2 * x + rng.gauss(0, 1) for x in xt]\nprint(split_conformal(xc, yc, xt, yt, 0.1, f))"
  }
 ]
})


## 1. Update a Beta prior  `un-1-1`
*(from the lesson: Bayesian updating)*

Write `beta_update(a, b, k, n)` returning `(a + k, b + n - k)`, `beta_mean(a, b)` = `a / (a + b)`, and `shrunk_rate(k, n, prior_mean, prior_n)`: the posterior mean when the prior is worth `prior_n` imaginary tries with success rate `prior_mean`, i.e. `(k + prior_mean * prior_n) / (n + prior_n)`.

In [ ]:
# your code here


In [ ]:
dsq.check("un-1-1")

## 2. A credible interval from a grid  `un-1-2`
*(from the lesson: Bayesian updating)*

Write `credible_interval(a, b, level=0.95, grid=2001)`. Evaluate the Beta density on `grid` evenly spaced points strictly inside (0, 1) (`x = (i + 0.5) / grid`) using `exp((a-1) log x + (b-1) log(1-x))` (the constant cancels once you normalise), normalise the values to sum to 1, accumulate them, and return the first `x` whose cumulative sum reaches `(1 - level)/2` and the first whose cumulative sum reaches `1 - (1 - level)/2`.

In [ ]:
# your code here


In [ ]:
dsq.check("un-1-2")

## 3. Thompson sampling  `un-1-3`
*(from the lesson: Bayesian updating)*

Write `thompson_pick(arms, seed)`: `arms` is a list of `(a, b)` Beta parameters. With `rng = random.Random(seed)` draw one `rng.betavariate(a, b)` per arm and return the index of the largest draw (the first on ties).

In [ ]:
# your code here


In [ ]:
dsq.check("un-1-3")

## 4. Bootstrap any statistic  `un-2-1`
*(from the lesson: Bootstrap and coverage)*

Write `bootstrap_ci(x, stat, B=2000, alpha=0.05, seed=0)`: with `rng = random.Random(seed)`, resample `x` with replacement `B` times (`rng.choices(x, k=len(x))`), compute `stat(sample)` each time, sort the results and return the values at positions `int(B * alpha / 2)` and `int(B * (1 - alpha / 2)) - 1`.

In [ ]:
# your code here


In [ ]:
dsq.check("un-2-1")

## 5. Does a 95% interval cover 95%?  `un-2-2`
*(from the lesson: Bootstrap and coverage)*

Write `coverage(true_mean, sd, n, trials, seed)`: with `rng = random.Random(seed)`, repeat `trials` times: draw `n` values from `rng.gauss(true_mean, sd)`, compute the sample mean `m` and the sample standard deviation `s` (`statistics.stdev`), form `m &plusmn; 1.96 * s / sqrt(n)` and check whether it contains `true_mean`. Return the share of trials that did.

In [ ]:
# your code here


In [ ]:
dsq.check("un-2-2")

## 6. How many samples do I need?  `un-2-3`
*(from the lesson: Bootstrap and coverage)*

Write `margin(sd, n, z=1.96)` = `z * sd / sqrt(n)` and `n_for_margin(sd, target, z=1.96)` = the smallest integer `n` with margin at most `target`, i.e. `ceil((z * sd / target) ** 2)`.

In [ ]:
# your code here


In [ ]:
dsq.check("un-2-3")

## 7. Expected calibration error  `un-3-1`
*(from the lesson: Calibration)*

Write `ece(probs, labels, bins=10)` for binary predictions: bin `i` holds probabilities in `[i/bins, (i+1)/bins)` (put a probability of exactly 1.0 into the last bin). For each non-empty bin take `|mean(labels) - mean(probs)|`, weight it by the bin&rsquo;s share of all samples, and sum.

In [ ]:
# your code here


In [ ]:
dsq.check("un-3-1")

## 8. A reliability table  `un-3-2`
*(from the lesson: Calibration)*

Write `reliability(probs, labels, bins=5)` returning a list of tuples `(lo, hi, n, mean_prob, frac_pos)` for every **non-empty** bin, in order, with `lo = i/bins` and `hi = (i+1)/bins` (same binning as before).

In [ ]:
# your code here


In [ ]:
dsq.check("un-3-2")

## 9. Find the temperature  `un-3-3`
*(from the lesson: Calibration)*

Write `nll(logits, labels, T)`: the mean negative log-likelihood of binary labels when the probability is `p = 1 / (1 + exp(-z / T))` (clip `p` to `[1e-12, 1 - 1e-12]`). Then `best_temperature(logits, labels, grid)`: the value in `grid` with the lowest NLL (the first on ties).

In [ ]:
# your code here


In [ ]:
dsq.check("un-3-3")

## 10. The conformal quantile  `un-4-1`
*(from the lesson: Conformal prediction)*

Write `conformal_q(scores, alpha)`: `n = len(scores)`, `k = ceil((n + 1) * (1 - alpha))`. If `k > n` return `float("inf")` (not enough calibration data for this confidence); otherwise return the k-th smallest score (`sorted(scores)[k - 1]`).

In [ ]:
# your code here


In [ ]:
dsq.check("un-4-1")

## 11. Intervals and their coverage  `un-4-2`
*(from the lesson: Conformal prediction)*

Write `interval(pred, q)` returning `(pred - q, pred + q)`, and `empirical_coverage(ys, preds, q)`: the share of cases with `|y - pred| <= q`.

In [ ]:
# your code here


In [ ]:
dsq.check("un-4-2")

## 12. Split conformal end to end  `un-4-3`
*(from the lesson: Conformal prediction)*

Write `split_conformal(xc, yc, xt, yt, alpha, f)`: `f` is the trained model (a function). Calibration scores are `abs(y - f(x))` over `(xc, yc)`; `q` comes from the given `conformal_q`. Return `(q, coverage)` where coverage is the share of test cases `(xt, yt)` with `abs(y - f(x)) <= q`.

In [ ]:
# your code here


In [ ]:
dsq.check("un-4-3")

## Done?

In [ ]:
dsq.score()